---
title: Comparing OPR bed picks with BedMachine
date: 2026-04-12
---

This notebook compares radar bed picks from OPR with the BedMachine Antarctica gridded data product. We load bed picks for the Vincennes Bay / Underwood region, fetch BedMachine via [earthaccess](https://earthaccess.readthedocs.io/), and visualize the differences on an interactive map.

:::{warning}
This notebook needs NASA Earthdata credentials and is **not** re-executed by CI. To regenerate outputs locally:

```bash
earthaccess  # one-time login (creates ~/.netrc entry)
uv run --extra docs jupyter nbconvert --execute --inplace docs/notebooks/bedmachine_comparison.ipynb
git add docs/notebooks/bedmachine_comparison.ipynb
```

The published rendering uses the saved cell outputs from the most recent local run.
:::


In [ ]:
import xopr
import numpy as np
import xarray as xr
import holoviews as hv
import hvplot.xarray
import hvplot.pandas
import geoviews.feature as gf
import cartopy.crs as ccrs
import geopandas as gpd
import earthaccess
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


## Load OPR bed picks for Vincennes Bay / Underwood

In [ ]:
# Create an OPRConnection object to load OPR radar data
opr = xopr.OPRConnection(cache_dir='radar_cache')

# Query for frames within the "Vincennes_Bay" region
region = xopr.geometry.get_antarctic_regions(name=["Vincennes_Bay"])
gdf = opr.query_frames(geometry=region).to_crs('EPSG:3031')
gdf.hvplot(by='collection', hover_cols=['id']).opts(aspect='equal')

In [ ]:
picks = opr.load_bed_picks(gdf, target_crs='EPSG:3031')
print(f"Loaded {len(picks):,} bed picks from {picks['id'].nunique()} frames")


## Load BedMachine and compute differences

We use [earthaccess](https://earthaccess.readthedocs.io/) to stream BedMachine Antarctica v4 directly from NSIDC. BedMachine bed elevation is referenced to the EIGEN-6C4 geoid, so we convert to WGS84 ellipsoidal heights (adding the geoid undulation) to match the OPR bed picks.

In [ ]:
# Fetch BedMachine Antarctica v4 from NSIDC
xlim = (picks.geometry.x.min(), picks.geometry.x.max())
ylim = (picks.geometry.y.min(), picks.geometry.y.max())
pad = 15e3  # 15 km padding

earthaccess.login()
results = earthaccess.search_data(short_name='NSIDC-0756', version='4', count=1)
files = earthaccess.open(results)
bm_ds = xr.open_dataset(files[0])

# Subset to region and convert to WGS84 ellipsoidal heights
bm_sub = bm_ds.sel(
    x=slice(xlim[0] - pad, xlim[1] + pad),
    y=slice(ylim[1] + pad, ylim[0] - pad),  # y is descending in BedMachine
)
bedmachine_bed = bm_sub['bed'] + bm_sub['geoid']  # geoid-referenced -> ellipsoidal
bedmachine_bed.name = 'bed_wgs84'
bedmachine_bed


In [ ]:
# Interpolate BedMachine at each OPR bed pick location
bm_at_picks = bedmachine_bed.interp(
    x=xr.DataArray(picks.geometry.x.values, dims='pick'),
    y=xr.DataArray(picks.geometry.y.values, dims='pick'),
    method='linear',
)

# Augment picks GeoDataFrame with the comparison columns
picks = picks.assign(
    bedmachine_elev=bm_at_picks.values,
    difference=picks['wgs84'].values - bm_at_picks.values,
).rename(columns={'wgs84': 'opr_bed_elev'})
pick_df = picks.dropna(subset=['difference'])

print(f"{len(pick_df)} points with valid comparison")
print(f"Mean difference: {pick_df['difference'].mean():.1f} m")
print(f"Median difference: {pick_df['difference'].median():.1f} m")
print(f"Std of difference: {pick_df['difference'].std():.1f} m")


## Interactive difference map

Points are colored by the difference (OPR minus BedMachine). Hover over any point to see the OPR and BedMachine elevations and the source frame ID.

In [ ]:
epsg_3031 = ccrs.Stereographic(central_latitude=-90, true_scale_latitude=-71)
coastline = gf.coastline.options(scale='50m').opts(projection=epsg_3031)
region_projected = xopr.geometry.project_geojson(region, source_crs='EPSG:4326', target_crs="EPSG:3031")
region_hv = hv.Polygons([region_projected]).opts(color='green', line_color='black', fill_alpha=0.2)

vlim = pick_df['difference'].abs().quantile(0.95)

diff_map = pick_df.hvplot.points(
    c='difference', cmap='RdBu_r', s=3,
    clim=(-vlim, vlim),
    hover_cols=['opr_bed_elev', 'bedmachine_elev', 'difference', 'id'],
).opts(
    clabel='OPR - BedMachine (m)',
    colorbar_position='right',
    tools=['hover', 'wheel_zoom', 'pan'],
    active_tools=['wheel_zoom'],
)

bm_bg = bedmachine_bed.hvplot.image(
    x='x', y='y', cmap='terrain', alpha=0.8,
).opts(clabel='BedMachine Bed WGS84 (m)', colorbar_position='left')

(bm_bg * coastline * region_hv * diff_map).opts(
    width=700, height=600, aspect='equal',
    xlim=xlim, ylim=ylim, title='OPR Bed Picks minus BedMachine',
)


## Radargram comparison for a single frame

Load a specific radar frame and plot the radargram in WGS84 elevation coordinates, overlaying the OPR bed/surface picks and the BedMachine bed elevation sampled along the flight track.

In [ ]:
frame_id = "Data_20090129_03_020"

frame = opr.load_frame(gdf.loc[frame_id])
frame

In [ ]:
# Add WGS84 coordinate and project frame
frame = xopr.radar_util.add_along_track(frame)
frame = xopr.radar_util.interpolate_to_vertical_grid(frame, vertical_coordinate='wgs84')
frame_proj = xopr.geometry.project_dataset(frame, "EPSG:3031")

# Load layers and convert to WGS84 elevation
layers = opr.get_layers(frame)
for name in layers:
    layers[name] = xopr.radar_util.add_along_track(layers[name])
    layers[name] = xopr.layer_twtt_to_range(layers[name], layers["standard:surface"], vertical_coordinate='wgs84')

# Sample BedMachine along this frame's track
bm_along_track = bedmachine_bed.interp(
    x=xr.DataArray(frame_proj.x.values, dims='slow_time'),
    y=xr.DataArray(frame_proj.y.values, dims='slow_time'),
    method='linear',
)
bm_along_track['along_track'] = frame.along_track

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))

# Plot radargram
pwr = 10 * np.log10(np.abs(frame.Data))
pwr.plot.imshow(
    x='along_track', y='wgs84', cmap='gray', ax=ax,
    vmin=np.percentile(pwr, 30), vmax=np.percentile(pwr, 97),
)

# Plot OPR layer picks
for name in layers:
    layers[name]['wgs84'].plot(ax=ax, x='along_track', linewidth=1.5, linestyle=':', label=f"OPR {name}")

# Plot BedMachine bed elevation along the track
bm_along_track.plot(ax=ax, x='along_track', linewidth=1, color='red', label='BedMachine bed')

ax.set_title(f"Frame {frame_id} — OPR picks vs BedMachine")
ax.set_xlabel("Along track distance (m)")
ax.set_ylabel("WGS84 Elevation (m)")
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()